# Real — eólica onshore Alemania (SMARD) · E_02 entrenamiento (MATLAB)

Llama a `psbp_fd_iteracion_mv` con el contrato del E_01. Si las trazas ya existen no vuelve a entrenar (`FORZAR = True` para repetir).

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
RAIZ = Path.cwd().resolve().parent
sys.path.insert(0, str(RAIZ))
from mv_psbp.pipeline import *
from mv_psbp import datos, convergencia, evaluacion, comparacion
pd.set_option("display.width", 200)


In [ ]:
# [CONFIG]
NOMBRE    = "eolica_onshore_DE"
INSUMO    = RAIZ / "data" / "reales" / "insumos" / "eolica_onshore_DE_smard.npz"
EID       = "mv_real_eolica_onshore_DE_v01_r01_K12"
DOMINIO   = "reales"
N_CHAINS  = 3
NB_MAX    = 12      # tope de funciones B-spline: p = K * L entra al gating por distancia
MCMC      = {"nsim": 2500, "burn": 500, "N": 30, "M": 50}   # con n = 3007 las cadenas convergen con 2500 (ver E_03)
LIMPIAR   = False


In [ ]:
import subprocess, shutil
FORZAR = False
paths = paths_de(EID, DOMINIO)
trazas = sorted(Path(paths["out_artefact"]).glob("chain_*_iter*.mat"))
if trazas and not FORZAR:
    print("trazas existentes, no se entrena:"); [print("  ", p.name) for p in trazas]
else:
    matlab = shutil.which("matlab") or r"C:/Program Files/MATLAB/R2026a/bin/matlab.exe"
    jobs = RAIZ / "artefact" / f"jobs_{EID}.json"
    log = RAIZ / "artefact" / f"log_{EID}.txt"
    cmd = [matlab, "-batch", f"psbp_fd_iteracion_mv('{jobs.as_posix()}', {N_CHAINS})", "-wait", "-logfile", str(log)]
    print(" ".join(cmd)); r = subprocess.run(cmd, cwd=RAIZ / "matlab", capture_output=True, text=True); print(log.read_text()[-3000:] if log.exists() else r.stdout[-3000:]); print(r.stderr[-2000:])